# Day 1 | Python Foundations for Bank Data

This notebook supports the first day of the AJB `Python for Data` module.

## Focus
- environment and folder checks
- reading CSV safely
- triage summaries
- strings, mappings, and cleaning functions
- clear notes about assumptions and rejected rows

## Reminder
Use synthetic training data only. Keep notebook sections clean and rerunnable.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/day1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


def read_csv_safe(path, required_cols=None, dtype=None):
    df = pd.read_csv(path, dtype=dtype)
    required_cols = required_cols or []
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return df


def triage_summary(df, id_col=None, date_col=None):
    summary = {
        "rows": len(df),
        "columns": df.shape[1],
        "null_cells": int(df.isna().sum().sum()),
    }
    if id_col and id_col in df.columns:
        summary["distinct_id"] = df[id_col].nunique(dropna=True)
    if date_col and date_col in df.columns:
        summary["min_date"] = df[date_col].min()
        summary["max_date"] = df[date_col].max()
    return pd.DataFrame([summary])

In [ ]:
txns = read_csv_safe(
    DATA_DIR / "transactions.csv",
    required_cols=["txn_id", "account_id", "txn_ts", "amount_sar", "channel", "status"],
    dtype={"account_id": "string"},
)

txns["txn_ts"] = pd.to_datetime(txns["txn_ts"], errors="coerce")
triage = triage_summary(txns, id_col="txn_id", date_col="txn_ts")
triage

# Basic inspection prompts for Exercise A
null_rates = txns.isna().mean().sort_values(ascending=False).rename("null_rate")
null_rates

duplicate_txns = txns[txns.duplicated("txn_id", keep=False)].copy()
duplicate_txns


def clean_amount(amount_text):
    if pd.isna(amount_text):
        return np.nan
    cleaned = str(amount_text).replace(",", "").strip()
    if cleaned == "":
        return np.nan
    return float(cleaned)


channel_map = {
    "ATM": "ATM",
    "MOB": "Mobile",
    "BRANCH": "Branch",
}


def clean_channel(channel_code):
    if pd.isna(channel_code):
        return np.nan
    return channel_map.get(str(channel_code).strip().upper(), "UNMAPPED")


txns["amount_clean_sar"] = txns["amount_sar"].apply(clean_amount)
txns["channel_clean"] = txns["channel"].apply(clean_channel)
txns[["txn_id", "amount_sar", "amount_clean_sar", "channel", "channel_clean"]].head()

## Day 1 Exercises

### Exercise A
- create a triage summary table
- quantify duplicate `txn_id` rows
- classify the file as fit, partly fit, or not yet fit for first-pass analysis
- support that judgement with one material-risk statement and one confidence statement
- export a `rejects.csv` file with `reason_code` if you attempt the extension path

### Exercise B
- improve `clean_amount()` and `clean_channel()`
- test the functions on valid, blank, and malformed inputs
- return useful messages or visible outcomes for bad values
- add one markdown note describing assumptions, failure behaviour, and output expectations

### End-of-day check
Before closing, restart the kernel and run all cells once.
Then ask: which conclusion in this notebook is evidence-based, and which is still assumption-heavy?